# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Turbo**: forma con 5 pasos + FlashVDM, textura turbo y Blender en dos procesos a la vez (archivos y videos). Barra con **porcentaje y tiempo restante** y **monitor en vivo** de GPU/VRAM/RAM/CPU/temperatura que se pone **en rojo** si algo se satura o hay cola.
* **Ves el modelo antes**: el estudio se abre en cuanto está el GLB animado, mientras los videos siguen renderizándose. El visor usa un **GLB ligero** (Draco + JPEG) que carga mucho más rápido en el celular. La GPU se **precalienta** al cargar y los tiempos aprendidos se guardan entre sesiones.
* **Esqueleto detallado (66 huesos)**: dedos de las manos (5 × 3 falanges, nombres de Mixamo), dedos de los pies, ojos y mandíbula, colocados midiendo la malla. Animaciones nuevas **Manos** y **Hablar**; los dedos se relajan o cierran en las demás.
* **Por etapas, de más a menos**: primero la malla completa de la IA en alta resolución (`malla_alta.glb`, pestaña *Alta*), luego se reduce a las caras pedidas para la textura y el esqueleto.
* **Física de telas**: túnicas, capas, mangas y faldas se liberan de piernas y brazos y cuelgan de la cadera o la espalda, para que no se rasguen al animar. Los vértices duplicados se sueldan y los pesos se calculan con hasta 3 intentos.
* **Huesos de tela con movimiento secundario**: en túnicas, faldas y capas se crean 4 cadenas de huesos (frente, atrás, izquierda, derecha) que se balancean con retraso e inercia en cada animación. Queda grabado en GLB, FBX y .blend.
* **Piezas con nombre**: Cabeza, Torso, Brazo izq., Mano der., Dedos pie izq.…
* **Texto a 3D** (como Tripo): escribe una descripción y se crea la imagen con HunyuanDiT y luego el modelo.
* **Dos motores de forma**: Hunyuan3D-2 (Turbo, segundos) o **TripoSG** del equipo de Tripo AI (más detalle). Blender **4.5 LTS**.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Cero vértices sin peso**: si alguno queda suelto, copia el peso de su vecino; el informe dice cuántos vértices mueve cada hueso.
* **LOD para juegos**: además del modelo completo, versiones al 50 % y 15 % con el mismo esqueleto y animaciones.
* **Visor web** (`visor.html` en el ZIP): botones para cada animación, cambio de calidad y modo AR en el celular.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden (la primera vez tardan unos minutos).
3. **Opcional pero recomendado:** ejecuta la **3b**. Genera un modelo de ejemplo sin página web y muestra los tiempos. Si esa celda termina con "LISTO", el motor funciona.
4. Abre la **5. Página web** (o la 4 dentro del notebook). Verás tres pestañas que se llenan en orden: **1 · Malla → 2 · Textura → 3 · Estudio animado**.
5. Para no perder tus modelos, usa la **6. Guardar en Drive**.

**Para actualizar a un notebook nuevo sin reiniciar:** ejecuta la celda **1** (cambia el código y conserva los modelos cargados) y después la **5**.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vttu4zYQfddXEMpD5F1Bli9xXBcq4FyaBkiaIE2AAkEgMBJtayOJXJLyJi32u/reL+sMScXybpLdfakAWxQ5PDNnZjgc3/ePmGaZpqSk5OTyhjzh4Gp+DgNWFksG/0QwuShKUlFFaKk5+dgwktF7FnneIa9JU1MyH8QxyRmZxuTkAPYOJvs4gBnEUow0inagzud/np5fzDzNc65ICb+K5wzfIArwOa1Bg1xSXGZ1a5yxcs3+Cs0WwRRXHmwQkhnh2omiyiCj2YqhAYIuixpMf0/6OVv31arqWZYVLUsKUNTT7FE3khrOZF3kjBNFSwAzCiXoKZusAKrOBTTyfN/3ikpwqQkY4UZq1eii9NpPoezn1cXFNUlALmL1upC8jpZMB/78cnSU4pofEr+f8VqzWvepGOV+z7u4uU6PTq9e2wbLX+1KweYipwp2H5wd/350/Oput7yFcA98cybbN4B4Xs4WJF2KJujNPAKPlk92gI8jqbnMVpvJhZ2IsianUaFSuqZFSQG0xWgfAcZ1RMG4FMJTZCwVkkOa6IKpIO5t7ZEMwlQTEdW0YiG8Nde0TCtWcflE+mT47t0oNjvYY8aEJsfmBZHb6BZUKa8D5h9e3oAj4ih2hCWt0rxQD0FV1OlCMpYu75Opsx7CfkilYHBgILW0pPf0A8e0Mzm3STBVkBWFdFKCQuL8jN81DwngZjzC3PnKnagJXGJTKEIDUjgzSxb4Larfi4xQl6fzuZn/JSEdk7e9nQP0M5DNFme8vyUH6VLRB5YXUgV5CG4slE75Q3ItG/ZiLEAIFX6v03ObkoLqVfSBF3WA+Q9p2BpjdXzbio32TeByU8naPLMpsoZogkqbxHanmbBHM4KDoZvnDAp6NqG2PJyJRlmjYZRmvKlBBeGSDM3yJy4fQohalboAPqcPHCAXHWMERGc/nll37Nh6aYtln/wGH173XOQF8LClMvFtrYQU5ZkGFcneYAgKNRMq2YtDkq2a+kEl4xifcCtENq9oxlQy2DPLBCodiP40CcGDeMyTYTyehlDTypTX6CXj4xdg+GJRcponv9JSoV+xSKYSkeMp4CpaiRK+huPQuIRJlYysw+Ea2bhgNJ11HDBGB7zOfH523eE9RkO/4j38Bu/h/897+ALvYex4n43JcPwm6/Pjo9MO7dF03NIeb2hP3mQ93ZC2PF8jbVn9AOv9YYf0YPICacVmX4DtkOsxGey9Sfpg/sfpYYf1cG/Ssh5tWE/fIj1+hfQgxuD8AGmTCd/DeeCO+A7BNqhqshVt251lQ6VpYrgooGeAu6Lb5MB9AbcOVEXoOWruMNa8XDOJLQdjEqVA1l4Y0Jdgl8NkVcBVbZoQVy8N8oG9sR0MCCBuCYpI4JZM94Ua36N9IegEwyoBaPBuu6teZBDErY8lzLRPqTFY+Xe2rmEiT+JWCroudFf6iRXLld4SGsdt6XMTcAw7l8Gt7zxo9lT0MehOhcQVDhE1IqeaBRi0TTnH61hCGc4D/ArJoBeSrel2djvCWMcT/LPxgxotk+3ivUFoZxBm674Rz1eNymRxzwLhbhu3Hix87FP/Fre7YPTu3WcS4NiZjd8nBz3y7z8mTXClu4Dz27fxwkcZNBolMIAg4lpoXLFDXIN5e3LMvB26+S8hV1RB94Ji5tzs3s3Cz9hsQ1aBdNsK4zqMEUM8vgTT6X1QtvUpbIBL/D/sVZwi", "engine.py": "eNrNfWlz28aW6Hf+CgxSqQAJBZOS7TicMPXsSHb8nmxrbCX3zlVUKJBokrCwBQsl2uX5W/N9ftmcpRtobBTt65l6KlvC0n26+/TZ+3TDNM1XSZFkM2MFf6zcNo5+MSIvDD3j5NSwfivjXenFJ6dHx8YD4zja0vtC3BVl5uHls1DEvsic0ej3PDF8YWRiWWZ5ks9GhvG9cZn4SW6E8D9KfIF//yqF78XG0svWHr4TsQFtvbj43fDwYis+GFYqslUQGq+e/v3lqzfGA4BkGE/PL9/YY8PLAyNOjI23M0QO5TyEUGTCKGOPhmDsEEwerMsAXgiHunEODaUCemXkwkihi9h6LNt++/SVYS295UZg/1NvHcQAFS7Pg7i8s41Q7GCMCfUCR+Bly02w5a6nXuaFMK6ZIYFm0BE5VqPwMh+6ItYlVM+5J2ehUWTewnsPr0WUJlDd2HoI6oEvtg/yTWRYfpAvE3wEPbPlABhF1PukEGthLJMYm1p6sQ94/Fc1EYb1K5T7gapCmQwwI2Lquuoq1dRRZKxFDIhcBklcz0aNt0zkZVjQXEHz6xIHRQUR44w27qoBo4GJyTeIu1DNRo2292VeiBwxlYoiyLBIFOSRLChyI4jzwovhn0ickWmaowARVGCHgaYy6KuzKoHwoChM0HKl3q+X1VWYLNQ19iQMqtsgUVfv8yRW10murjKhrvJNWQRhdberiuTlArC/FHn1pNhkwvODeF09CKIKzocgBSoWI3Ufl1G6w57H6WiVJZFx8fLckO9eRt5ajPnPmzQfcQHHqDGwCtaj0W9nb8+MOfTaSb1i4/hBFnuRsNS9t8jxr+W62LDr2vYIptI9f/Pr/4NaVWed82R5Y9mjP16e/e0dvLBMaC0uzLFhLrzlDf4NxYrus2C9KUx7NBp9Y7wDQs4CoLYZdMtDosmNdQB8BLNqJMbKC3McXGKkwOJAEkUZLYAfRAh0k20DP8lGND4H+Np9+erpizP34uXfz86xC48fupPJBP+P8O27l6c4zuPJwyfQtC9WRu6thEvNWkG0tmdEVkAjf3ghdAmYPk4yEFvBB4/okjuIM4ZvrcKLvDgZI2PC7xUWLRLbQRpDOMEKqgOJ5EyBS2pDzoZDv2WD+JN5AfABtFuKsyxLMssEnpYyAUcPyNB6sKX+AQax6u3Y2MCwALiTBx+EavvW+Nk4OTaQe+hqX1vnFWSBMioCeQisCcIN5Cq0akVBHEQJgkvvbNksNfE9AP/F6J+AL2hwnYHgEXUDkXdn4fBsaETNYA0WRizHvUzSnWXrL5xiA4QSe0FoWarmuIJhq2k4f/r613+8eVc1iDVRzvLMxUDEb188Q5KFP09Nu7/teCuywpJFqEQmQKDE+FoS2jIUXlymblIWaVnk1o0Q6fzJBGgniN1VJoS7Xsynk5oCn4GQ9UjCaZIS0GRsA/EehWZA2qpCneSVwjNIzldEWGS7utN+Dn3OgfWFb/lIsIaPg0T55uCviuXfJ0FssXhw3vx+6Z6+fAs4+N60bUSSKhXkICos364RjyPBJkjWgSDJb9wyR+5qwrIdKgiq//vvTyZV7dsNyBfsJBCBYYHmsfwcZx6RhXRMlX7WUaZNCI0P2vahc0lqTezGG9mjLAKlLix/bARrYG3hCiTHfH6ZlaJZQRumuAtymDIf9J/pgPg1W63iD5TNRJRshV6sUerLUSPuliItjDP6A+q0bj31QGmMRssQ/hpnMZgYgl8iybkuMG3hulYuwtUYNfxq/jqJQR+EyXqeZkFcaAPBQk4KHcRyiGzZLV8UYllovEUFAQIUhd/Nx4CCBb7AZppv8o2XiuE3brTd87LYW3HgderBAIffuDdBjOQi2aR6+T5ZIJN8NL1lgeaYOTOAS00Ru8sk9PDuU7OCC4Yf8OQtVHoOuqpu6xtQ22iLAdsKP0AOtfACzJNkG4Q2WU4g/UC3gzpbgjlcSA5uwhfA0tgfUi/Q/hMH+yNtZbg/eUQPFmyowYPHE6fTRwDiIi1rWr6fxWWXHbRmNAJuSBG9a06Z+l4hLCwPVOH5VpIC3zZbtTUJMUzMFUE32mBTEBDwqU1qnr/r4JxeSVS4aZKEUGK5ci7JRrmA+7M7cCTAMbFAs7i3SXYjgPtPQI0xZVyZ8pl5XXcZ0BF5NzB1Wd5BGMkGN7mREqRmPhHnYFISIjQuA6n8LkDbZZn4wTpB0xforEQLQ1rQmk+z8yp/ZgwvozREWoA3FTivyIIFDAfso1Kg70A0FaMTALY3OkWgPUAKLAOPDONc5IDySjcokcmz5frBEsRF/cbJRQFj8UD7WKbkRFCFyFD23lLIk/cUrHkQlas5VAy5EQrs48ahqoovoTrRyFA5wCm18Hn8NdiqonmA+TmcNgSPyB/7N9gikECa3IPt4jjolAAF5ztrAW91fjFBwedUsqVZr5rFrv8ZxhqxeDz6J36YMyp2g7lG4ePeCvQscqnywCVM8jkMX4DxHRcPtLCDSXPceRxtYSoa7HoOyriowg2Vk44uPXn+xHG+eO9lYRV2AK+1wWPkdW3KNejn9cpbCndTLpQPloOr7dIsA/9ldRWwhFAPXV0Py2C03zK035owLNuhgXeNFJjyjN65Adl9A8Uq2GJLpRxkJBQcA2Xr/v4AHc6LzFo55CmS7CdQK25uS8/z60FtgH6s6NE0YGlY5muSl2kJfkIoBAm0Kr7y2/MZTGnLhpNW+GRUswXQSuZb1LFmMzEge9J4chuAviRthsXRa11AE4tytRJgO63nExt7u+rihM1YVAj9+FpAU6zArMcPjZ9/BofU7i0oHcjFMNrlCOPeAjHOB1rRC7vGAJpEGE5w8Jdm1tFo+zl6+phGKu6a/SiSwkP9mpeRJe5ARaZWBoY1zbDdtRetlXnBHCNjWjJMB8wyMz4yMGn1zpzp6pPx4hm+NrtDW5kftREYR0YxcyarT7mmRSReCKqmlFk9pUEqNAkxRn9+bECFRaIRBTPt7sQHJ5VtTfRWJddWIuM0uHweJrevvAIEQ7y+ANBhoBmc2Bbg6L7yDraGGqsAVzkG76xFx7KXqyQECTyn/pa5cDF8AVIs77owQDo0oOaUfWM8B1dh88cpoNwXaIOsgiUFSDk2+x/TyR05mZmXBjI2OSh+1PAcEXugG9wVgt76kRUtXS9cJ3MzWrZ8IMnul7uU4wBdgL2tDLbUZZsDBEpXsBgVWtB2AtcsTeIAGuoRKQfC/2zYkmBxlDW9kj3NhHoL7gHYVTTL8o5MKP0BTbj+4C80LIsdeX1NrfYrMqBubDrGJdY2rEdog8PzH6p+2xiiAZMVo1wUjIWugUuokRZFZcF6MnIvpLBtFZw2fgUUYYiZAztG4UWLAApQuF5ZtwC8AiVj3Bx2g95MuDvKW5K9+JXCX35DyapQaQIqupZ2eOdg+BFsl9xZlr4HgqqIytABgk9u3WJ1cgz8iUjbUymOP7f4AmyLDTh2N+3SDbHC/rLs+TOAsM4SQNxbiiRkOjPr89lvoTVKzCuzC/rtJrG7Tkuw2WDyUKHU1HK/XoC2Nde+t+WG698ZRRNW3TQFeRpO/jBw4iSiWaAqQ189klTL5EdSy7Mdx2nJnV6p0gowsBOkaYh+s3FQFQ/9mBtZ3T/yg+Joe3w0OSIEgADohp2+QL4wCu4TXD0k0qDLNo01Z+eL5oZnBVxX5taeeWmEhg6cgD6Edn08NZxoSyMZJjuKH33B6ACwpL2tlwUUl80LL/8C4pMRrIOGD/7J51NgLxFG269LhTVC9lFia1qsBsmBdBqYGLs7a184Z83Z2keb90/b/+yk9U1Zl87/idlqzNMYXasyRyWbJ6CAp7R8OjB7HFqtpoRve+ejoe0KcddnQl9g9Y7hTD0uKRZrXWm4oMZ0KXrdUisC8ANusw0WTH8987rjRefoj9O6Xr4Hb6s+oqLeG9bH76j177Az1SVB5A59xxbNd596KG2v3Tv4ohPo7kfofs+iQ3T9IrfhevT7qveb9hoVmTx1EkOzvV2itcOvioHeQdqj/eDVegH0t9+pB7f55mu5I4co8/0LHbrpVvEn2n5dHq1XCXIBTESBT3Atgw/sJKMRCQ/1gHingSszWa3QWzGvZ/d0VPlwZOi7y7R0ZdWWM4dcc4/dI9OFXj5t4suwliDBMlzI2oGfgg5GnTlimw1xhhjps5S7gbaUVoMb8n/cVlf6A7s3/Jbq00AStHqim6MDLrBTJJYJKDPt/rWQhkFHb269LCpTqzca80ouM4SgCjkUc29YRdV2I4HOd+Woyma6ix2/xx46cJ6PSTlxADY6eHWc0TSjJY0gZDf0RmQxTDnO1K+/nz41dkAAwboeDQVfw3WSBUXE4ddlefr6NWduAYw0g15nKjvIw+QnD+O2mPoVJsvDvEUZrGk4Bg2jRCcdKiyVnnKs2BUkLzPIXW/rwfCAIq0WMbCzf6/r1Q349qX3nGbebZPtI4DH2Q2xuJVpCWPwpafHYwN+2XA9GRvyn213VtArsA7+soKoVcQRYRikubCs6Y+Px8ZDAHNyAhfHk4lN8b9wbsH12Jj+iL8ewa/jR4/aDQHVAnPG6xDh/ATVpz8hoClC/LEG9BgeYp7E9HEvGHQ5GesvKN0M45W+2AZLMTdxHkwb/P0YDEwQbiD6W4FWCnpy9SBeiQz1AkkmC9rn515ZJEsvx6UYBDc2/AJU3JxfouAqpo/t/sCYRRku8yAaY5KWW7eQFyLN5zAbyRLTEdxMgM1Von6YT4+fDNiKCGIJ9sxNPlcSi2/N67HMtYNOzdcAlXJMXOonLg9FIt+0jduaUPNdvNxkIDo/CMseDRg+GMIBUxmUJrLyIaLisyL7hgIeYepgYiRRUAS+Mj8rMYO6yS0SmbBVpTVEKUle4c8nTemDGiLBdFKZakR5pDwJgVdloZ4Gl2MQ7suwhCZpEVazCyhRkn/eCZZcurjBvFJcZWXj2THO4iLAVTIjEu8TXJtZh5TFhsHepCmAVjBphVcUmRyIvkh3mDENFXBcLYMahtNvTA+6RlDDsArClcrKGvLVocnaztJasgDXZFMdyXcP6jJH26kzPToNVqsyF1kOV3kBrC18rQGeRZS9gA++sZ0clzYt+2p2MplctxKvVG9kYcy5AYlRhqGxSPwd+HWYgpoJHwgkBLPDWFRhKVPSCtgiFl7YnSAsZW3pMViX1ljztVo0SJJibj4ASinQirzEl+9emC3a46eG9cfTd5fGA4Pujacv7ZmMlQKRULrWKgA1jPoJnuTwkD2v+5VVDwHpK8HD9p4cZTOG2VY1uFQIw1RpcPmO17J7LLKdSgaLc8yCmzB+7EMKNhbIsRYMIl/CKIrctO0u7ctpcFJJdXl1pWZIoUpiv5cRCNQiC8Bm0YKvcP/21bMXTfXQTpeRfax9CbXg3Msr2Ac3k2FRCb/jjDTg32KiIZQ6mjoPAQNs77EWE1svtIZaIduvMeL726nIlprpTJdSd01FNyBRFLGTKWlYEab+I4FXlF5Rv2nfS4uHag418o7f0+1Xb6ijFvPDS8ecXVQJA2kUNqQB5fSSRmfBAsoYFAmm+zayRZmDRcTJ2/s5Wz5kzd1cOSDx7Mp8cVUSpjn1MpnI3FzRVS06r71I+Je8NyDbPYdHVl6uVsHd3HTSeI2GjQhFIeYcYKKl7Ra7U4KxtxXWauNggrjdSt2i9DL5bliygMlPWYZal+UC+2Ltkuc2j1PHyzJvZ11NMfdG/boGowzZyY1FMW+xmJaCGsTgybUardMzael/9IXmY4/O2GNCtrpQaS3stLQNARkt463PUsRmMRNaUtkxGMF7A4qKAN0czao5Ma+lHpLiUNe/GI84PvWjA54AzC4mmeVXmrpVK+lMjM4l/7USKOOAWQw2pgXTRU2coGuBc5ejagzWZYIBRZzH5Gp6bevZcb2RBuAfuNS5ZnVvgLHtSvW6TM0ApOMHK8JvEVAkIqO0JpE5ODB5pzjrFTx6S09GA+EMmdelDwYnWuXhD9fghqoK0qTnp60QpVZZdm+u9cySaV66G9EG91nxZ30sc5qSbm9w9trj/NJkniUuCvOWDt5RgRpDJuC1XQAVfGiFGrr2i5SMOW266XnTFLn4s0W/mSs42yDDrExsKsl21pDPyevJVMbFBeKlh5n1tspjoSyMoZAA891kyOPCnU8lBU4+biMHLGe/kR3zAJ+2kmYmnDTzX//ZlzTDiTPox31cSwiSSL7bZl7krhffXWN187NSdb9GJp/a9aMn86W6cmXJ7S7WnOPSkA/oDewRDE3jAMu3NuOAj2F3tFwVP6HsL9wY1PY+mtuGRj1136Q5bhxYgbXgxWB65K2iGw+oIUw3ntxFQptO5nO50YRCg5UaxIpX4JGNjZNrJwL7zTZ+Bi0w0QdWIYmrJkXdQt/46jV7q72HRTe99+9y4cnPxBIYCvgXaDFZvBe0WRL9rwx5mrcTlp6fedK7jsD9FLXIRgT0jrQqsQOdd5dzsTiJP4gssTxQXdNJI5aLmW53eWs278B0uJuOjR383U0Bxl3OGBzTFShVuNpVz3byWQ/GCAlZkloWwtwRXHA7pwQX/uo2Qc+GLGJvfCh1Oe9q+h7smukTLcD6V3/sDm3Lwbhd/hcoB7ANmKItKzeOjFsQQiAYsCrcbfiu6+pCVdAaqICt6eT4IZhZ8LuzOYoUMSH8+CGLrfwvzShOw6Bw840QhVLjeA3uXYYrRrwLcIxb/8bohY/l5r+Gt3wabMENph1um+Q9uQy8Qsu7YWOBYZQcTUZKG8QspFUS+52QCrW8VyBo+wiy9cJrssIQ57a2ElEbnc19ctQoURhJmgyhd2BgDfFZgwmwXxUXENqrckva0eA5XrzTY5m5WOcYw8oBidgDLysod1guF481NYNrGcHYoMxeAcYmWp64Bym0OwGAbSVKGHLPikarRc49C1oLORJVDG4/qM6GDn2EjpcCLn0gcGwO2rEb/N8Hua8aygkcr1b7b4RU8u7AUm2gFfGY8yIxBULwERIalAPGQlMZZuhvwMgTZ/Lwui2PsDgJ6xmJSiAWY1OCvMxnmNCGO1zBTAYbPe/kImNtYiCH+Msyx+2QiOodeSUB9OBvYArE9phEDDxAmVQ95TTsgNKwvRj0VmzXnUWHjUAlKuZGpRMs3e5DXSspi+a+HKyDoEDosNi1EcAHgEcNjKnHba+oLK7w7XVlA6PyRw7QhK2U4fQUhRV5E51UCMngAPGrmCUUu6g9FvRXAxVqQ09hjJtTXUznzw/NGrZOhfT1xHOoJxPkcCB445dLuDnQUH+ODpfIJIgaCdgzZKLGa8u28HmnVG93OoUHbesdlgNrtvEyQkKqXjnoprwTRUujRsCUvu+SJ9ksS207KCPBB88Z0w4juQMC3wXx2pWmj18CkaIF7qrq1p4qOJsuWBMuOBwBJje4AuZsT5VlCJrA3STArKjA6QIdpOPJZLKnHTfJcE2DCcVdJhvyPMO2T4G+R4TJo3QgAWPG7vVbWlEhbTrbPnpEeBR3MER4RZZORLjUHshI0rzFS5+5elPN4UCkTfasJ87Wm96yZ6hquIBhsCKHZpohMzZ4qp1VcOdyDk7eou/BCmGoprtTvhc/x/vzFSjixedh8PoWO7fHDSVWyRRSlqgENA4wfqnfz/ooQBMkko1rKQW6fl5LrLbUJES3NkpIYYeb6PIqxMlLlu1Qp9w+gUchgMPbdNE4GoyujQycsnfzZYsXvfQiR6+v28sQLfX9Sh5EcT0Up+0A/EaF06uM9gSMQYNZmabwgcDYmjejHZwYcPcwnaFaUhsNbNThKemaOJ+dwqf6t8Kww70RbWXt1whCPWV9xsza7SA4AfhsAul48H3BmKgMC1xm/HgzA6MRrYobabBSS05QiAg4nk1UjVJqS0TlmLIZRfCQeaa1HQ0UyKmFe1J44MF1tZFG5hXuTwC6to0/21a06ovMS2x1gXbMaXSrg9zTeKPl4rqdwRKLu8JK67wlbqudf1TTB+4lInRxUsu9iU6ayMopebmvXrGv2opqtjQKEhDqQfDFKdKO96Cjnths3qsnxs9z8IkYJ4++LJsrKTYia3W4idNW57tj6c3vknBbOV6Np4iq/vwuKtaT41Vtf6qXA4eSOUSUFjveCmq1ozjMVtBJOduEv6aA/N9ZF/knUmukpG9k1+xZNJEUNGhTd/NvsBY/tcd7U25GX7T4YjdXe4Y71pvC07MuwwqFZ/SruD0y7t7IuVEHGSjHRwp8NC2/dMWmVxMGqwqm8S/ztoWASw7KQOgxRbtrSlX/dEYgP7dJ9deYKs6Pbq5Z3/DRPwPHVpHiIU2E2+dv7D1btL8x/uD9BnXeUW5Yq4wOSPsBz5rzQmBbx7ENzmMEgxZ3WhthYng+WImi7S1XfrJHkUTCYsUJuYo74DVpPOZyvAXi+WL7Xp3Mh+HbxgYKtPepn0DeY945wIPrMfXv7Xyjh7U3zxtUvg5xJ3JTK1L2jdg1DZgkLXRXHgWNPO0N5f5UE6gVjdBZZ028oRrVqaMbVdubib1RJ6rcOEAuJC/t4UJb0AmLXYEO0NVsZvz0o4bBTeNsFr+M0tzCIY7pBCgXhi83Dfc0JLG/cTbizg/ALy6wgenxdW0GZmUs0ccBXkqxUgcMAcPIS+IZeU1RUL7uRlTdveHi9irQuGs5yEZYessbcnbkNRCx6gfmf02PTx42oSghPacTP6DdubkpcT0y8PEIjg2l+synzhNMcQLRJOYTKNicfCiazM1fPbBhvMwc8wNUMKppXmFXXQIkF7gc2DMmKcrkft5MhIHYYotPdFOaHPh1hh586EUL3zNQSM8MiYrYxShj4IU9SKeeScefigI/rxXefBC9oeoWe3BztZnGbJ8apc666c+sp+T0XtHSyPm/qGTDqG0PVpoc06wvrky6bhzQQ1MuS8obLso3ell2rrkoX3NJutYL8gI3FsMrLtTaB1FNrixY33Px6l6vJClAjUveyZHxXaMbk1Z2uJ4zSTNvARUeg3b6tzIoKK2TVkZMvZsgjfgMKVJ4My3SKnm3Yi5bndaoFyI+1Mt07RbWkXol4uAmYGZlvRA90Ut96l3N6ReXumfI1rm+EiVXY2rRYivHsddy10V2v+HSQCdaCiDnR20IlVExYPx0DkJ8Tkmg6lxTzJkj/alNH8psPNcuWEo2mDc87blyuFnW0W8WdvC/nevab3FWgq/21lH6wf9K6vGfSvDxn/EAflAG0m9d/lVXtRCUf/vB1LJxkSShVd3a40o08gu+sWsZySa3vLMHoYMIVZAXiT1WApGe8bVdC0J6Km9sJRE52FXBv1EiDC0L3abQqeSmolY+u4b8w8Z5ivVLILIrEx6BPBiyzd6qwyExSEWV0PK66U1mbACu5Uj7aEq7tcKz9xgrkkx5ka1IOpnf/vu30be+++1v37769p1r4ibMm/6D1Miwa52e1jqskQ71bLSOB9Xi8V95UfpB4myKKDRt2vCgVUY1ht2WGg1FqyWVI69O+bNW+AOPpmoP9IJPrHIJ/eOhUWCRwWHQelhf5Gr2mTbhltMumym81PLKJDvc/Rh/oixOW9MPZ4ochXak4DfGWwxDk6gJjelkYnzLp3Wgx4E7GfB0aUzJQlPCy4U6PBuP482CJNNOvUDaOLuS56ahD2WdXVXnpl1LZw5ZU2Y6ITGcVceIUY2HNa5cdQwArqPCq4nz0wQPtq4beIBtahVYQauKVY17e8FwGkdCFcHyxgIHZoHyconLTjnlBiVpa65M03y69eIPJLEXuFxv5CXMTUQOnY9JLyTE6dhwRl+29hpZCoS86cCWr/r4KLKfoAOYNZbjAlqXLO5krAwG/mhsWK0dOVMcKWWN0oimPW5EZUF4OHsLqOXhivEdjb8ngx+7c+sFBbT4yG5iMAVPSXRRuIrbkgtgNA6NPtsC0loYoETiuggfSWUVmDdUzHGuxgZc5/OBKQMO98CgiHtOdQXuoSwA64BVKCk1V3GrcG+CcYWfvLPYKdslrtWwVp02elUdZ3iN4eCpri3UOdsg7Jc3mFrYysHTTMATEIxnMWaIeDN19j3wLDoiuHqRyKN1zFaITrXQo2DanTvSO9cqpc5nbA1hELnxVp4c1hB9JDI1eblFQynv84DJJBriIBUppMCYLyPbRKCAJ2AVKTPGmnQ5dN3d5HgomtnVqUcnpybyj9k+CsQkP2hbh2HMgV3m3R8C91//ySdqmK11AABzaHdZ7830cyr2rfp8ydEiLRuytRLYM3OVpG8JrHxYPlUTZp4HURoQ+jHHMOOZ8Nble5Eludm3kgdykA83q05QBUuJjG4O+9KCrIWRIUxAliae3DH+8qm9d7FRZYUwsTWNijqxaZW41N68vZ57+CkQe9NKtFXf3sobXEyjbsvooL6k/PPceDzBH6at+1aQuaw90I4j7rBHTUuFLD2T5oDw4KzDhWl/taPbeHIJwfqyeh8pHNu4UJrR8HC3XVriOjtMdI47Oz38mIcP9XPDotPGVNQTvwkCphHQTmLfS6NkiExIypiESNwPSjbBx4oSZuNP0OxHXseHa2rc7D1foZsB0LP6/1Wo5MAUggHKpT72zT7bqaaXLQOYp6G5H7T4+6q3DP6G4W8xoZlcRp98YOzUM6az+rTDHR6IDNbt5yuX2qocOLSFX/rQixtpp7L+aRDJdMxWLOmhymT9TCFsXgRxofSRHqd/+fRzTx1qKAu17tNe8elj3N6jOgbwMJQBwaqBy9yrGBTcfRTH5o5cEQo+gIv8pcQ3BGk/HaoJbVKiosPjGU0S7Xc7XA0M7l05eCn4sL0djT0eh9m+faZgw2DUPDl08h41nbsf5CNpHYy6nMaREiKjZvMNf08DXT+sgcOzkeYLq68brUpBpziH6sNHGDEgCxpkBC/YNr9tREfJOBqkUzrYFnPpkrz65NQM/ECwKohGvers6DHaGWSR8+4s/LgMx5tqgMVi6DjGOG1HKu6hUq2TF/XhnbxWx+dp4oGbAfr3PHMzSlPmHhlxIHtJJ7PslnqqMhE6raeaPEZcI6V67HhLEJy7oopcD349QQ+8tQopq0wWUnG4OsKdYziSQ9x4KDyd2A4OIVyb2CkfLDO85s5Vm9qp/3XEWXSgfGo6uQKVvSXd4yYRcl2gZ6JBdLu1J2Pjzh4NGA0o8cDaBx/+R5B1dI8+eKM2eeONnmyHe7Lt9GS7pyfqHSKLyudrnUHOOSgEXEAxLjroAuRk4FFAKJ8hBcloHH+fyfAWWZDVhKXBErH6yoVyS0NPMQca4c7Xif9Vx41bDathD4cA/osSXIPuJvh2eI1DetXCbzu2kS37eXN136dloGZPfEdHA5Roj7PaNqofzO/k5QKsYKvrp8tSuD0gHfO6uhbgV5F9GQbXAvdVuL61PKni8VXcHSdGiQHilGYFZV1YUj9WNrXUkPpyJcv66hZdCH27xqrzsRg8U04yd8XY7UwM/OqcFEmv39Ayi0en9dOH/tZo94B4F7F2fjGeHRKjp9myVb4BjxOroB9p/OPlBdG1X9G1RslVbw+bqP+pyRqcMMbZmOVJ24Plb7MRHlizoQ5QKrPWcW+e/d+x8e7yfGykgfiAh2rylxF2xvmb03rO7v5/JNeCDDRe824OfsGfOFoJh7+/te9cLir7w9ww/4z/jDEWtLrr1vqslBuFZephy58t3EXYNZIXuqFVx9g1e0gPvCt7aBEOj6pKG7HOxkbPV30Agbem/fnbhVVQpLl5KOjYNRw3gEnH1L6k/UWirhAN2sF62Ujr00RQTIOS3u1v1qUZOKTxu3brd31tt/0nGpr+qRZlJfI3X1QKT3DHn2ppvrW7y+Eua74w8V2OKuBxNcSWZv+6N9cI7gYcDuje1Y2LdBTc4cU9nQd1SZ27urYBMwVuu+vA9VzqqicblgPTamK34K200qiRThl74HgPxHCDyhQGCRsUfZnjPv0wt2yGWkVl0JKgHFnS+XMZOlarzZ4beHN2P8bV+fYuuS9zOlfKKlSQcjoQnq1qSYhVPYIJteoSklerEovwELB4HICq0pQOE4bfXfXuhSi/qTrnbDH+bO784srkKzpvLi3xAWVX1/iuJQYiVQqNIGVBgZIcF8jw3DtOGHK9fBkEKgNJEtEc989quQ+4DbLLnhI/KCwRMsJ3brMA5hllsd2XI/PTE1wgiVIPzA4OmIDm1nj6Awmgsqg+39fyQXAfIzX+Ie2BT2fUmOd4/lHjMyRV5oYfZHOuHqTzD+mYiHPOeLoRu/mN0mH8Ldr5aqsekIU+Z0O95QaAOx7kG5etGgtq2gfsltlKzVYWXSVVy8XtPXKR2mzLRY3x7hG8+yTp1h4QRts+oFvbHhJdhwjW7f2CdRDiIdKuX+ptD5B69wk9rVtXZiUEmBZI86tW1LuB1SddeG67RRpM/TV4WmOOK5K+3Nl41f1eQfUFpgNEAMjvvkOrKu24kQKiMtK2DUlxALN/0UYtno+eJYnBUJpyySlocd0+1rflHMoPmrKf2fdNU91mWe3JM1ltaUsuNgqWQxhizp/VkC+apdcJ+B7amwZ+VCoSDDET+dAhw63leCxJz/4PoilYRqLYJH69c1GfOe18oAilO4h4FO5wYzYXweU3tZ1/BCmdjAYliMDH9ZuXF+7p2fPzp5dnp/wxRhT7IfgR4XxKhPehm5xIYQj5uV2gYDwdDxO8yJvtzjsCxU5qLb67fPP27JSmzsGPy2BnLct0ovQhGnby4DZTBTjYacKP9P7Zf0Y7RuL6RtQp/UEyS5fvVngyb40B3qKCd40EqUyA67yUiEQektn7vwUFnd4c4kFOujt59IvhbelIMlDVDmAUT3jGJCTpelL9Vy/PzwAlr8/w294f6/NDZNhhZlzVyz+00K/8qJk8b06tk3ChfuunGZlw8NDkGgxZRgSFAl+q3BAk6PzdrvpWIIF7ooMDRluWIcHjEoOAyKPO1AEoCOmkOT4qwEd5D0KJPPDC+BRccoERDJ7eXINRLwZRw+E6X2WTuB4e6u0Tph5hh864AI7oxfkzQ70+GN6tWEhqdvDAZxOB4EHgWcJfgIL3hwMDY8VZLe4IWKtzz5/9/QA40nOC+j8iGVzQLZ0+sykB4YdAUBP/I078K1z6qcjh4HGAN4cgnjwGEH/wBiGRM1pwsfr8zal9MDApI+SpiuZT+YX5mhVNW1uClAGiHs76A180+ErkSxEPcFUvBR9rcCpeYhoegiLXIoisMGZMjIDjeKuWUrAn1Gk6YLcqtx/eOgD6Onk8oYnqgKveDgDhHiuM/sEhRsWKEpmf6o0z0nBRJ5x9aXhLC2n1mFZqt8jjMUdD57gh1uQYq9oxUm0SoS0a7a/ELSM8C/5KJvw+Oz97fXqG3609IlVzdLTylkWS7Y4on65M6SGtAE3o8qK7+twXylfBQGTVdEeG3NERNxDg7BJ64AYwxHHiDlR4CZVNiT+4YxSadBKTxTcMlnEqX/CN3QeuCooy7tUT2sjBlat7Bqy2bfBLedcLupozHOJUJXbxI5ncxeg7qkU09XYomR1KytWwGmJ732RdgIywup0eYEQspiQa6kj9GWUJnrOq/2Ve6d3qGwnVYp5qoQ4X0bEwVKhW4uSdyIZar+rlwtqsoBOfm5/ETTP+Ios8xcS5IGcBKBcH4AO1zPWXLy/O6DlYpPrzd5enb36/vC9XAmM2cjvUolzRyS9T7XPLmLw4bx2zhR1GCzDNHO5NJ+/bfEpHUxmrzIsELV1QHVmXU1XZ6DOfYxy4x2qMyHoGvxQ9WCtrQDSsP/0fbJg5hNbre0fVSosUCvxVJ06hn1fiv99vrGtZmIZMWcjOCZjYtLnTwePWU2tqYx7yQ/o0RI+U7vFJ6fDYuOXy8HHf8iQvQk4mj6rqrq1JgcZfGBs2MXR0D9sOqlQP6lVDVjPPqF4R3Xn3LIq2IQ18xadDDFco8ez23HV7KA97s27EbqzSupsHv0X5ULAFnBfjlzlTNjYEMBQuhj/YJPmADh8b3tyuEU471xx4hTLMNbZP1sUd7jUAx91k3UHE0IjGp/iRapSkuJEVZdNkYGWFoV0dnUwmk1l3uzG/Ho1G7tnrFy9fn6kJGY1Qc4O8ckW8xu8dQMF5mgGhS/Stw2ThhYasNpLdUlA6e8Fq8Gc1vCoeITujgP03wB0viQ==", "blender_rig.py": "...", "app.py": "eNrVfV1vG8mW2Lt+RaWNu2xqmi2SkmyLNr3Q2B7bwXhGsT33BuEIvcXuItlWs7unPyjJgoC8ZIEACS6QlwUWCbIXCBIsFhfYYB8SZN/G/2T+QPITcs6pqv5ik6Jm5m4QD0bqj+pTVafOd506MgzjTZiJZMY/sUsxZearhHt+1GWeCJhIsxxuRnATZklEz8IoE9MoumARc6NlxGI+90O+F+fTwHc5PAvx/yzhqQg54wyuViKFT1kesiwPAQRcPw+i3JsFPBG2YRh7/jKOkozNg2iqrz+mUaivo1RfJTz0omVxJ/RVKtxEZEWzNJ/GSeSKtHiSLRIBEwvnxQN/KfaKjmnSjKdwtbc3S6Ils5l6B7OZ+XP1UIQwW6FfzUXmyCd7e6ffvHn7no3ZxHjOl4CRxLCY8TxKEkFX73mQe1xfZvLqS+4H8uqdiKM0wqu3PIxSvHjNp/jyfO/s3cv3Lz8g7Js9Bv8ewBosp4kYMTPmaZRaLHKzRAiLuRzQbrFMXGV5wrs2G/QZNYHFWgqACzCWEaxixD7kyTQicAZdGgDt2GKHj48s9rAP/yw27B897lqyzQuR8SDgHrV7bLHjwdBig+P1hu947MtWQwA3PH5osaP1VqeAAWxz1FddPl5v85Zf+UtqddxvdHjUP3kIzW73Xv7L07dnX79877x48w7mZhwg6QGtHrzOw+uch4cvesMDnqZAGgfiii/jQDgAdC5SY29vzxMz5sz8QKSm1x1Rr1HiiYRWUeLJ4SG09yJ7HkxxUdRT4JTGk8Sf27PpVfnEngYi9Cot5Kr4nwpo1GHxTzcDnuOZ70Z2NP1Y+bp4vMyCymMgxESkfhQeenaKbxpAY1984qkeayyAFvQNT1wfllTfwgQcP5xFNjIeUB1+PkOCmcAnMc8W9sfID03PYrMum0UJmzE/VPjyZ0w3Eld+mqXm+jfdAuQXAHNWgkiBk4RnIu/b+GPtW2Pf6Ha71YlBfzMURPg5AASZQHczYE8vvfSzhWkadhDNDUt3C8IhT7CxXvY0Xy55cm3ilNXac5gsYQDY2jT4xzzNBHLiza3sPAA+J4QUQzFnQKWAQ5Ro8PPN6Yjt798gjEmHeNFBVu+cj6xbyZv7++ynv/wPMBovd30PxaOBkyl7Lb8yukwEqWAGXHxR9DgzVAfUXHbSsVjnzzvdaic//g8tBNgNl03hvtO9ZfEVvjQqAEX6Qy4CkUWsAhioAVublUeLHKkHOuvDC3ljMaIoBsKwBrLyVaw+6vQAXhf7rrxMxTwHeZ46WQTiRc3jlqUVMp4ZX0XJko/YTWeZB5m/AvLinRrSqi8U0joDoK8s6tzWhmW2dT1D8NQrje6DQhuuZW1KndRvdKsw7PhltyF22dqNalx2VAX+DuhWJKPaAtAjjbgaQn76/X9nH3wBKigdMRo+2zyxEqe1yTCTpH63OSV8WMwG5lIlpC2zKnppzuzLgObR+vE0KObY/vGN2Vn5ngAC67AvWJolZgsU2aLTBS6BZikzBYgU+ZLFAShTwjbMqTbRtc+LKRs1XiPC3IVk9/fVGkl5U2Nq0iEuCGkQJ0rWFPLE5nEMaDCN07IRUB5MBoWylIEkUGpQzpU8rHUTRJ5DAmBzJ19/+wKezEUSrXVSUxyA+wsk5FUhVLoGSewLi61Q6sohlT2e234mlqnZbRvYkoMR4iQi8MVKbB7cO9lgxLA9itQQKRi0M3PRdkPzM5WCVpGd0dLXSiSgJgWSZsA39gVS5fMfU7RXASC2TBXzTTo1AFJ0rz7/rXxGrT//rxA+PHBhkAdLHs7BaDTqWrfQvoE/FWBZRqlUECAofZGE8PuaTRP+CSUnYI+B/IW5AtWCeZvOcxG2TUxK3Lum9Zpa6VkVk6p83DmHhUUcVuaBXU85MIsrpAUPQME64N22gZBEx9UE9sW5bRnNWaEdXB7AzxhtDh4Uw2qCkuiO8zDDOfizz3900TrbiF8z88ngnfMESZRlEeh0GD74GAga0ZDmKVsJ10ejeuNkfDIa0fwedNkzNth5QmBjyPUV1KmCU5+eekh4B6fk6potP/8BaFok+GkFxSXtpn7o4MeAGiQQRLJmuu1DG4HF32e/UfbIkmyT1AeSYhy9Oa3oN+BTa8gKE6yPBOehX7PiqZ4GyohLPVYl/UAzk0czOd9MKT/99d/87//5e3ZzeasAKYPN+D78PlQSij7pggH37s0r8oaM0zxDV8PgOS0eUD8wJD1Z0BXIdvLCEp+TvIAXbs69JI+FRx+8h5EWaMHXIXiIOYC4VXbiyheXInEW2TIwvfE3IHwtthD+fJGNH6Ingu/V40s/EeOvOKgRi4G3FHP1fJnO6UpNHtzdl9KtZocvmIl+qQCbu4skBB6yP0v4EshWJKCZKiIPxGwMUMkFB4sfLB70I9B3VtSDBrAH9jhd3GGNG8qxt3FeYFyXy6LQDivy1PNXoHOvAzHuyBmPbuTv2/jqieenoFyvR7NAXD3hoFLCHimAkSswmvBkncIMtKf92XVPuWi6JULoeYA7NwPdNnKjIF+GT+Y8Hg360NGUuxfzJAKdO3owGA4OB4/bYE/JD+mhF5+no8ERfAiAomT04ISfeJw/mUGnvRk45sH1KL0Gw37Zy/3OsxZQ1XnTV6n/SYyOHsZXnWf/5z//t3//9AAaPMNWz1CBmrC8iHXjfT7FEAcnwxMEfJwHKWdtHUw1eDVE8VicCNF59kqA1OUJkgXK4DenAP6lJs6nB9Nn6AnA93IA9FNzHcYm8iQAfQMLnaRFgOKHPMoENfkBGGZmZGNg8MzE2IeNP8wu20eRASZ9QUhI0iVB/IDu2sz4s9X4Bl+UzZDcG82MPyMeGBRtiA3WGknmQOTR8Ey6lxNJE5fGeSADMg6P/QN00Mc3TRLuVEkYLLA/v/lBja1CwpqZEnfcuYGftx2N+kvfyxZAYf3fPFknbklNo36D0tqobJ08OyBqg+gSiCcPghQsFxE+YVdJL4Xhg+7rZQl8AlKm0wAuPys/ghWWw28QaZ0z0RLqEf+NEhz+E1CAcz/sZVE8OkKSfcoZSJhZMf8M3ots3AHrm4cXnQaBGnXKfDx9PJ3NnpRsMDhEmKfTxE9QXsU8pLAQRgLjAMXTT3/5V08PeEmde3sPWO8X/GNzxRSoc4lQUoD41k+XkXR9gNfQfIIxgC5h0cxHAwP1XxEBAh8pD6XdDLoQx4nq2uUe2hwEzpSKsves8HXgslALXZs9h4bE255YwcMVmhWlFAZmh69VHPQB2ARsCeYWrDMqXRBxFA0EeFxOwGaodeB1mC59DN0gJlf+KmKhD82RcSlIBPDtvX/+7Zek6rQ+ci65n5lpZrEQSAP87wQsEJGmVeUSC+ya0ALKAjqa8o8RAxuL7AAcK42D/QWC+Au2jHCs6C5Ba75CI7DQK5cLYD7qSodb0mxi4D2Y/EXcBZ95gF/jvOR1PTATZm/27ZMTi5rNwHWxUKu5Y/Ax4ElniQYFhgWqQqmHjTtZHwxCuz9D77WM/1C7NBAiBrjHhfWE0MGkqo4h4T6Iwnliv0ySKEGjH3kMrE8EDm2hZyJRGRCKIxPJuMRkuoguHT0PJ8JIFaxt6mNYnKMNRIsHpCZD5rjw3FuCGizVspTCfghr4mYywHiBccomaGXeUTMbDLYQ7RVhwtC/zLMsCm038N2LLgp3wD3QVFoVdTfr8EZsgpMB1GJs7kJ6tyUV4WpJo0ZN9wH7Ts0PrTXSQUSSRC4r8WmExOGKIA94onwVoC8PrcuQLcUSaJZXh1S1m9AAMl4SySlu8IAJPkRIcWn2+Q9I/z/+3SESgbaMVNT1x38cAVZTNgWVGm72A5TsOpUBTWDHsyj4/Md5hAFvxYphYQ+XK66MI4pOSizEsDbrkcvqvOKWWGfcJU7QT0G8oqzEx8/YMZjjhHu0AFXHYH3EDo3FBMUdAi+jFrFYIGZwTWIcmGUhhP7lkN6BV4DnlXCmc4tglIH/JlpmHIxz2geAlkLALJDBHXk5z30PmZy60tYs3EQZzwRatJ6I1iDSU3ACAGpKYXQEDwjJ+DQQescBQUofHkiwuAYLmF8IFWhZA+zRxoIYf0hybAkLnYyNQnYzM/n8B9xQ6Bok65ZxNjaMUuyNgUPOtKTpVoQgUdsAqIeHn3hVDF6XMrzwkID+wC1Ax41WVFTN6gJxpVjBOyAUuRVl4y+0qXD3Yn//cACyayBJRoRzaFbuEZmFrKJlZ35KVEHEQ+usn5RdKbvelFMnO9Po2kDQfmxWrPYWYdewR5HV3MRXjzy6iV3/8z+EdkW0FkK7b/cHSkyD9yRIOwCiaOskVE4JIhWWPfrpX//XqnROruvjknMdIzZsbO9kkdyCUZOSFFoCEFeugKm+pF8oYDmo661TnRnfkECKcxglhoqSjWMF2S9ujfZ1KPuoDhncHD9zaHnMdZ7sSheXPlBgP/pIHGo/0s6iCxE6C3FlPlQWbkae6wwkNCAZN3LQ7TyTjnQovVJYAngovUfSacUNadkRk15mBSdKJYPUP8dWfuWTrA/XFd16S5+hXTGBoZ7jWLPCdY8CDzVRAFxgYpPuZNQb9isqFR/acRSb0NKiLkCcSj6eEfmASAM2rtCm0vq488ivTHUHYjWIeAaNKxs7+G4pW5Ywo9ABbQvUIu0drwEZp4pfeLWnykDR4QW8rQzzMkouqtyzRrIIIiELR9JAkoc/W1Jv01iqs6ok16KbvDRx1dXyuxDad8LD4AhRJUl3+NHdJOMrQv3uUW4R+nd+rBTBWK65uqOAfKkupJZElrgTHG1SjBFDhLsuezoGH7aiFPDibjBSIY2nUQQmSqmhAF2a5sb64m5gWo0RNHXTvfszpe9Qnkfp3JB2bGLSY5TzPMnkfqbxAZsUe4ILqSCN7hrdaoYoaFg9uo981RkS4KkKJPi1TsjOJkf9JruOwYPv2o6DXOY4t1LErn1CwOijAqwtHTiww1yzPpE4katbfFa+noH5GAQtDCsdEICPloRk9iLnw/5AV6ZyflEAgCjhwKQhGR4K1WoUpY9lSIux4mZJIaLwWyK2iDlIKQGC1aqHDlWk0Fj4tMgV21P2QVu+tBWvl9gN+LWhoopkHG0nJhVUUYbPAVnTb7V1g8DRX42CnHS+0SBNUKV57IFoMFNwed1MeGM19UZD3BJr6QL1Bd/fZ+Z2Jw486TNYWG1NKGcb7Yd6L7T4aBR0qxazam4ScnczAIej0toDE0ClvmBwTQErLD1Sy6TcUHRCFxW/ct1AaNpaH5RxCY6RCGX2kxc9YapnroMY2tSqEJjeydqdxFrJCsAYVp0GhnKBPuitsrtXPGtvWq55HeQ9Vr0RZDqLQl9QwKHYkbhmle1NaVFWlp6Y/j4LfzgqYVtV0NhR4i7AtU3/n6+9g/NynLW1T6riW1KDRfsp6HQokrDwCp+pFiuyOkNYGdPclq+jkm2gCa4egDK9empNkV9T0z7SUjCko2vM/SQ6fNg3ihQcelzzp7vaOERwn9Cv9jB2bX/yY4l0SruCx+bkU3ze4lR/ipUgnJxj1LtI09rMC9BlG1WrOHGTqqshEKt+t9ewEqyauJaoUCiwl/ERQJazabLNT//x37D9/bdS6CC6kULuiHPt79OGF864lqFUhw16ywcLQQ1Ys8qUZ+7ClEORvr2y87byjHIx6bPNVE7e5IonPlgL6FDqgNwGHxeWGcYQRPgSXZLJeeFi+JakRBHmS2AMWCvqutsSOvTZAZjaoWpQhA2/imiLFXA0uD24KRvcVj1ZICU0UGi3N/VDIGawns0ZRk4SRVkzG02WzW5AUjH/ER4hddxmXxOex/SzzkyIB+2FAMMi9Z/XWwCOKrugQDHFTseUMmiBTPCue0uJY+svk0kHoHbOa7Pfwcar9/uf/t3Gfqsus+I7ubq1rdlorqlwee2Qnk3NUii/4oEA0sFYqdTBKTg0PPFo69ykzC1YdbGCJpGMV0DLWYIbhN0yGIP7i0RPBS15d+cPyuxd+9vvPmCOqEomRAcN47fSsuquxVs0BD+VErKOOhTyfpiXlOMv5zsJX5SjMFVHTtWOw7lBAgWfkeVIj7qtht5OKZY1qatzHZbzhnWPeNRrb8Jri60tvQeO/+D43E5EHIABbhqopQxW5GEqQiBQk9HD/rla/AiAyuU3xSoboQB5T5L4Bc+4QiOglJbR+zWXcG8DrvQKkobi4bV5NxKL9brTeyvSQHdbV5WKiooacTABDNl+6Imr870W2rv31r1OfV/kmR+U4RC6td0ovq7D0jcwFFpyh1Ss43TX+sCoixyhtEFuZAipJi/VK8wdtsHLBwsDKGF98PUkY01NTXFVUQU8Tf//tG/+tCbKbmZJDdp9DS2raYFssDoWaNhAI0WJn5C378vPhLZ2ntYqB8TMcSFm0JXNE1cUOqZOi5IR4rTGCDqoEYH5XzxcLWG0sqG98pMs54FDu1nXlYgECnf6zHZzj9vw3kGDB7FiUhSr8tJPHb7CUx3TAO07QixuDfQryrnNYKonLtwoFHkqaG8KO+7eor59d/oWFPJqaeep8MA8wi2H/sgegO140LI9BqCgrUxiLRqjofnqS8zuXNqxSDAP5/Y3lCX76uw7lqfc46N2WOZ8MjgHo3U+6QOxNHq/oZfNXmopvSWoYvOcUudc7i4EzAttOWHTXWHPtNoyV2sZSxg2E1dIlM7rN+8/UJA90wHxeZzry1XCl/q6cumWLXALnq6L3VIZ/NS0tk5eSCArJBKUA/El/Ojb/XW6RAFZHkwis9IIgW993kuXPuv1wGFNrnsw2DGC9j9xnK8N95akSVp0fU2Lam3fEcWpoI2dJ4LAxBHIIttL+CX0JqNvYzddWWG0EBwD12GUA4+nxtZgU7oQQaA27FweI3QnyrM4z9RDDCToS/Bz4N34EENsHlwVgPG0xURGhK+kAL+iAyV6d0vuvJhgZRpdIDd9axndMpaJLg70EVsS6xzaWZQBuQ/XRIy458AnQ3wM4oVPDs/x51EJAunBya4yGc1UXw36wyMi64Mb/Lh4ImnauENpVUAauD0OXCU/uVParHiQknXgZiaSAc4PgI1XuIELvwp+BcTDawUKLh31HFUDLvo4i8t8zAvHkrhF8DpPuxwu8cvkwim2S65KkYTZhsX7yah30q8FoxCgVcwXrpa4EtoRqKmL0hdI+AwTr3XSi3nSpxjRu+hjRNuwXsR4MMeDhalMdwx8TKoA9lnwazC+gzJqV2Ov9sEAOgv+3dvBN9IHhWpZVrVcws6zt3JmGPLBPEi4A2lPfpLKe9rmGn+Mpql8zjOUWSEssoEP8aSRwd0MY1O4O4h2WOjgfPFOnUH6+s3bKoGcPFIUcjKUJHLSl6RxcqwI4fGh2ncEB0yeXjINJEglFeHXg+Pp8XTWx8vfoLFnGr8FLWNoWYktTqaPprOZbrHXtH5Vc916MDvxHk4r8J7L/lzVn+jz/mF/I7QPKH+1HMb2R7PpoXeClz/+/XPN/r+z2GuYDSXhnkhhixYGUIC7APe0HndA9zOru45yOxFZAzBssZyMR8JRSQsYQSiovxTjKzohuCCjcDHpgcR4NsaFqbXCPEaY6vHR4yOPQv70GdkD0GHVQcMXdQ+NhlvfvKy0RyW5aKSp0/QznLWhz3OANPsdyEGMoTw+IWFm3bwG3f0aHmJO1hVJyi7JNqnDt+oSFbq5qoduYPon/dF5w8SmhSijC0/jKLjG5G0Ww8CydNy5gbHedjDmFIw7Idi1mJeZRBfAajeYpLhlJDNDtuxRBie0H9rHdfQO7Ie3nYNnFT+aKKIynjTmoWbtmHu4QzQ6jK/YSZH7WeR4Dus5nhsT5286aq0PDzv14XQeDPnQO3zYuVUpyTBDnWE6gE7TKADfZjNgdxO4zrMbpI1b9nT67IbokBTUTX6LucJPD3CWGgsoRdA9AzEjMxa0ZCksU+iEGj1j/ZKsaoRIZmnAb8lgiwJ9+uZPitx1xNIgNyBWN11rqDKsb9sSvqEPFcJPCZMFjjpKFssTj4hTNGYJLLYjXKxjGpSVQ7SO9sRTuroajDv9DrsaArX+Dqj7Gu5vkA379gBvh7VbzQhqLsfH+lHP4+kCD7pcjztH7AhJnHpE0w0D6S2J5RIbz376679h7+kEhBc1j33REqu8csowB9RJcfl96bHQhmRtZVUPUtRDD3/1e5nQJ49saZTsbT5SUE2cHvaHw+F0nSnqCypH1Vz6ZnJ0C4lpKnwMgOkB5jNTKjMrjgI8aeTib8u5rp19aJ5qgIm7ojcV2SUmftPhhsuExyP8Qaca6CDBDS7a7VonLRjWNgfqZHD8lHVDGbPySdMVBBFw0AS82QnExNv4Ut7+Ti1b1YipDm41p93GL6MroGfWZ0DM7OY1kCweQRfJSpxS/uo7dFtKsb4h3f4RHu5o9iGT1xFFrA9I0rx0e9PpSGolzYLMCIPZfY0U2hurAR1osCTDureNkxV7e8/fY/I1WpvTyLu2mC2PJdBiczwjx25qBwAeDR4PPPbPpIfIwa3ea/kCXJKewsdxH7BQaf+E1cjwdu/BwkvYYsBuyhT8IVIwU5jqqyYxu6mfd9ENhohKhs1S3xON4U4H3vDoCVvjO8Vd+k3BTIhDprkJT+cgWMyP+VPAxeHa0yDCfCvVi7ytdyZlRw2HCrpCiOyztigPMDGwCgWJjCc9Wiv0ok76nphbyjC2Hjxyj7zZrNu2ULPZrPp4TxtLxYGJR40FXpdPjddXPZDyXnQ5QhYbHsF7NY7jYxh54s9TcIlsJGB2owj7lunnAZ/Cz5t6J8M6kd3uzaIoQ0rUTIKsWmuAFK/2FHM/8Jzc1w7cBqfmTG2VxSrvRiwpypXYdJnaX/IU8z19DCc6i1yMDT/0/DlmO4YizxJwhOlpGoBVuT36gLgdT0rQr6JoHoiv4KkpK9mgz2EUgh01XcrDtIfGPZ4+sFPQ7UUHyNdOSQkOmaSG4mT4tu294/HkotKoBIbk2QJN0ihCa2ugwelWTXC0lg7RG7Yici5hVd4WgFST+iQp91ZDkfJFz698pUHo942xEHnVIUl5U4wn87NAtPdV2SuI82wb0lsbbMY6nZBwUgEy1kP6alkAdzg7pEE2224f6V2gC4zvAH8jat2FcC+A7xV2W0avl3x7y43L3/isnRA2NmqMW/L7VFDSTKUsEFpoZxMKGTjzqUE+8eFjZTrKCkPyRJGfLVAwfInkkppEL2Pj9IxRngSeO8HtfRgQsfeYfsLo0nQMyrhVMHxMx4bZHT+78SIX/NIws5GobTfgafq1n4L/4nlmB2eBhTS6GO/xxLJykFoP6V10aYpALB3fGxugVY3Gvi80eQtQQDajd/2ANQf9fbgWtz+rbcvXOvoBo3/qKDHlczd6u/+5OTyOh4fQP/2Q+yLxoho43fdzOl9rpi6Y0+NDC0MBjnSlD4d9ixXzR/3bREAVzgc+Tc2W9402pqFOtRob2qqEDzzdQAoDk8uADE0iw7HxrTyr8PmP02rmXsRMqj1GSbFdw9p5p5Z6wx3tRRSA3BwbLz/aI8bZUoDiX4F9fBHigsgZpH6AbjdPllFyzz5wb2hsvPfpqFQO3EJZDZjMTQcU6BBG84CCca8edID/ipLnglzM6fze4QubfSWPO+KsPkZ0ThPsquDz36b2PWdBB+7HQzzAc+XIm6PuRgj61AQs4hs636GW8KtaPgdy9nUMLB/7wX2GQ9ii1IKx8e7Vl6dGcQ5/eNjfPChNiqeuC5oSsGIab7H6Am3OY7EY8HgCj2Mho0RlTi2ijxyPG+EOditnbuoD2fqOpiQ7QWy3oOmlHEkdP1umPRhumXaxgmLWtiRnNFmUFPav2x8didjcIbDcr9ihPDO13tlrWEAkOLnGjf7uBdzR5eCgixdJFJPovxMJE3keyKJTInRghNCClqh8IQ+O0BtsswsbqC/xmxJO0QE+opsmfON8h3MQPEDTe9OYtSAGTISqrghyiHEvnivKGDDznT8vuKuRfbWBoiT+36EfbdLJJEzBwxwIOfBTWfyDjsDKwa4dj7rzX6H2pB9lbKcNdb5Djuu5MpxMueFZcLOesFeYSNb9BbxSJC+EqiGERU0whzn2UVJFH+kGHnqgIPOA/xwdokvpPHyoirrRzpz4hN0ND7v3W+fTgDLDQcUF91hjyZMSne8DMDwSs2+D2jm22MB+JA+7juWRRIXepciSKL3f2N5i8QCclS5EdQ8RXzmntGXRVQErmj7lNG4oWHVHLw1EwKyHNqCibz8uUFFiAl3eEIw17t0PG89hiPjRPZBAp5KqzDjZcFBXHlZ6/4qZVOlIH4faRRi1Sab2TtT839Kg0LzAyPN9eUyylx4uFoYoTonwKRjR0nKh9+z0zR3LR2HQbE1aqfKthcBC36kY/hnQChWavLd00JJBSZeRrOhKJrI81Xw8GIJwGBz3Wb/fl0XcAIHgmYKPhJm3a+cydhUV1BEVBFgCyIQKvjLzYaWfJxWax4ojHlflZ39ehzQn+l6WorOZrA07Yseqvq2lRxNQHa/t6wQDa6j0iYH5HUi5WIAWf2OFWTxlIZcMk4vUMk5w+c4nh+fd+y6YWnF9esaMr7p3DPSehmVx6HSbjPoXuQ8LglE07y48KVxhfs82iHpCsmbR/aTQb2V5RzOKyWMJLFkbjVbxXjZ45QBnY6yqIpcarK6xpJq+PTv6WWq5ZMBvMC3FBfGDZ+xGFfmBZd8SuYXIrqkMtmBZ5CGN8pTVSkLu3XlMtUGvVGu6MIDKetNqkqqU5Od/CCl/hiDcxRP6WO+2labVYphjyjA9djPEaab0pKxeUgQB2ktbGZWwA3jEaMv5n2BewXxLF8TD2/NMq8WhKctUZmF3J6PH5xvhYtWqq+20hqdiZC3p1MQKGxSuTMcTsqDPSxPwo4A2G62UWpDkrZ/KWFK9nOJaz84aZk9dTCPzP9H6S7ylyy14m/NAAniFOjm51k7Th8jFsEQkK1JyrPAURKqsoc5PvgenyDJuFDxQHtcxhpeoxpXMkxeXirE3Al1gVqEc7FfgmtMpH8eN8jADAxBL/sIKlET/r96cVU6V7IL00xUWB0ExONr7FUQwHUav2W6HlGJjsbrqwBxFMuMGP1ODnFFFd/Onf/tfBn0870yFfO/gb1leoDa6AeaMPTxaG99Aj+9wWHp/9PmWPqhsQR28LtUuf9f7GOo+sKD7fdCgxvOcrBnQFWgG7OIH7LJ+sqwLNP4mX05xBsPDowID78XS12fBhUt1u8Z3hBCK0jFbNdJpIHiG5ZO2Ma2q/lBHMNbDBycJq4got+C4gPoVNCEDX1ZwwujjFviyLkQNem/wGODSDw0fcwu1GvVRmqc+W/ooLbIEtbb54993f52lKApTNO20IxzPI5Qkgz6N7+ioX1hpuP2gP7y3r1Gnr3flWfmd1Kcqj1HD4JH68wcwLPXa0FRfkpVyw5gs930/4+nrCNbMrB7KvJfJNHX0Gdw7BCw9xhga6DiD4nVGqeW+qhwG3dzTzlGcZvjkTsi/yDy6E/qauj2j5H2sMATIv/NzeQC2um901xf31XjbmHp99L8VCctlXWRMHjpgMhN4B7MhKUe2YS4/Y79KpK6AlfF32Kl6VN2awmSQNqWNfyIjTxuj3FCYdX9f2aT7+zYDvGBx7sRfioRKU+zop8oiGZbafqn4vNdY+pGqFlQjCupPiexqR+HWnDKeHrWomvpOHG5rggGf7rAjNygKd2D8vSz1sVlcLB2sRiIR+/rD26/NtfJ+pz/kn//IOBiPwhWgDSrl9nawxIbVShVyTEXEbNuoCk9+x0G11gDZaYRt1QnVSIvTe5tHiiPbMNCWzoHT1kSzYvkXwDI8wZLzJnC/nAWmXX5RVLVgKQhOqQp2orJWEbOZ2Fp8d10Vk3xdDvqN/fh3dTf7x3/cSTHtaCCs6G/ySNzQGMx1uV5uDj7eYqStHHBjWwC90h7udjBtMWZ1MgSLgp59BwJWHo24KxC+1HFlIo/6wZmK4FtGmzQWvNnmGuqzOO2y/mM0VYYLWoGmYahaaXKrY4aFdYNrR0Y35XGERvUlmCBYCRZDBaHtfGxX97L1QT3dGoNqWaUr2YHtLng4F2a1U4vp35P2amnAyc0SUaiv6BQIMptFotFScsySksNSlGRJOrBKvrOUJqEjr+UkTMCwLMRqliU8W7ls8ieq61m6Wr9qcc+6zf1rV/isbpmpCp+6luf5VuwBXZZLJhdQrwtO9hyP7oRunoDt7F6X0aO91nBE2ZCOk6GzgGXAREJW+f4+1f+V4Bsi2U5zFw9smtUqVGp0E0VJ1dHdZ1zsHj0TWat+SxL/JZ2VvSn5oai7KoAsVUcSnlVORFcaB3w5xQPDZqXwh1WeCu9qAJM5jp5iOhVehYe2PAhvlrUj9CflLMsPppWuqeANQJ5qtiVHw9JewTm9gs8taYpvWJjKAewK7OKIuR4MWcDVk5HLSFY6+PkIIBjg6OTCBDnLQfs6dxEqZuKQCNf5C0qqIqi9X15tncrlJnvO+5fvfvvyXb3y+AwkkoMpwib+KA90PvfxL5lg9WuqOI77nq4IPE7A0FjKqOzWNE8pwgj2N+2rBf40WauUnUbuhSjLkV7Iqil+iMlHuC1zKaZVOwsjK3qsVJX0oqUAioc1vLCSQ72OV2vJHwIKFBClouHebD7uu1arImuOysjyMBS4I14ODssGtA8ss1WV7EyPAVERIyoSUo2IKYsRvr5gR/1uI5NR4tCWv6RjkO4w7dSe+lgQxujb9B9GulqMU10Huw05376nalHrsGsVczQEGL8irYDnobtw5DLT9MaPHj/UE6OJjtfor+AgeFdmqZd8JYGadDAloeKU48rU1GPqS+JT9q6OreNBK7EpjwTD1yLMHFle0sFE1uKz6NIRiAP1QGcbDdqinBQMF55Dh1rGk2ZZjOr2hRIYmqommiWo9C2y/rooIJ51iz+z6hUZ/Fei9pcyK02MLXVo4LNq6d16GQMQUXNBCU69H+jnt7inc7UlB8dYZFmcjg4O5kCy+dR2o2VlJNVBHYCRIXgq0gM8G5BmB+jVo+A92OykG5Xve4Co/KrHl97DI5SglOjcKBoEc3UXILtxlhbrR4+Oj2vSFR7XSRXkEP7t1RQks0cFyfMUs0k5HljW9DuoF1jTzevFbfDJWrnoPAlSPhPmSfefkgGqNM94ni3GJs7KKsbZsuONzZwlkAFWy23L4t6dV34Re0jFSL6sdHCGx8up0WQa1B41jgFEwy2V5l+Ti5JP6hxULVHn1qt5nFHBowkRUCnvjV4vjHr456lk0R/5CLrdxTufEZMAjwyGj2jdBqMbXKrbXYLbstrGuDrCN2cv0UYFlyOpPn//4QUgtVK1o1LMHYualOUKSz1DfxakXsgwY0/Zw36jyJw8CIs9qfIfNpJBUK97QwYo1rYUYApiHMNMCvEw4b1P/d5J7/yL721YnnI1vkeRgbFQ7ELWxW/WAFuuKyK54Esbj03EZotPDzYJL+sdbyogrE2+CdXvR/XsoHquzPN8S33hOlEqWjlXeNqlRoUsj2yUf0O7UYkChW9hZeCU14qYKgjfpfrvvogQs+KZ/CveUeWPmlCefSLTpCPbrv61AJIyTVupXSCtCSWSNVIQtIuazZKjlfTvr0tLcqCB0YhQ8rYs0ZrcUGoBWkvJXw587/8CTQMVTg==", "estudio.html": "eNq1PF1z20hy7/oVML27AGwQ/BJliRTos2Wt7Sv57LO8t6lSqewBMCRhARgcAFKkaFbdL8hLqpK8pe4hqcpDqlKVh3vLw/mf3C/IT0j3zAAYkJRWu5ts1S6BmZ6env7uHmiPH/jMy5cJ1aZ5FI72jvFHC0k8cRo0a4yOp5T4o+OI5kTzpiTNaO40Zvm4edgAYD4ck4g6jXlArxOW5g3NY3FOYwC7Dvx86vh0Hni0yV+sIA7ygITNzCMhdTqIIw/ykI5Os3zmB0zrvThuiZG94yxf4u8gZSxfNZvuZPCw0+30OofDZjMhMQ3h3e343X14D4OYDh52SdfvHcBrvsgHD+khPaIU3qIZvB2RI58QeCOeN3jYd/vuuA1v7AqwjI/8A3e992jlskUzC26CeDJwWerTtAkja+SJ5TJ/uYpIOgniQXs4pcFkmg867fa3Q5d4V5OUzWJ/MCepgZSaQ4+FLJXvQI05HANbBp39ZKFlyyynUXMWWE2SJCFtigHrnE4Y1X54bWUkzpoZTYPxkM1pOg7Z9WAa+D6N13sPc5asEpYBH1k8IG7GwllOhzAKxCSLYUjHuXhKJYXw6AdZEpLlYBzSxXBCksEhDJIwmMTNAHbOBh4IjKbDm2YQ+3Qx6A8RsnmdAij+Z71nZ3SyqqHZOjUXiTkUfBt08KAsDHxNzKKAislmSvxglgnaEuL7yO9eIkjrJguxnebO8pzFlu3m8UrZLU+BPQlJgeQal0HK5e7tjZ2eKBvBs9aBXYREgngKjAZMszQDVAkLOCeup8CYJmzj0UHMKh5IomxWI+lh1+uOe2RL6LU1A2AfcUPqrxigDfLlwO71i319OiazMMcFyQr5O+jAs+9Xok5pSPJgTvnwaJMp9xfBDhp9X7MjGs9KAccspsNtJRMa1RbK1tml+b9SBw7gOQpi4SwGnSOci8iiKa3toD2fVhZBZjnjxNssofHGCdyQeVcwi4MascSvkEMdZij3QkPO6SJvcrMYoBlt2zAH8KnHUsI5I/gkiT9EvUKK6wfEMyl84ktKLb1DBUviB1M8cu0IYmiHBhaLsoiE4WrLOoBvJB9wMYInoYvkFj1iV6WYVBw4jBQ3rwt5tAGNy4CkaIdPEhMDbmvcLfWRx2i9Y5ZGwo5Bp+nfGU2YMRXvU3M03BX5QUo9jh4ImkXxLu9VODbUGCHVo4P5NVBI4iDKVltOECVT93PDz7MsD8bLpoxhEnOBolCgX213iqsq1KS7YQc/6aI2iLJ30QWhbqcg+bgYeDgejwFVEI/ZDhGKaAJBi9v8AdK4rVSoERA0qZB0IcX9IZ67MN2O3YddspxMqEv8CV0hvj5KSwKgNSuSe9LmkgvphMb+6jZXxDeUavZkw1ns9zedhWqGMnPwvF/sq0pz3zh/zYeWzCjPYoN/0/xgXvO11Um3ZCzGaytqCrzDDmSSAsRo7XWBIFgVjq7i+Q5v1VNOEMRcgtKVPgTfsUsSQQw5IbiyGnU7qNptWtvqVLCst97zSDwn2aZP/01E/YBoRqUsB/twEHO1p/HkqOAMathh4XoOy4ToEPMLbSvBUE3vqC7VHl8hTASRCl0SWrd7AzWSyHMq+DoVPs1NNxRBK/yp3KDXFtA5Y2G2qru2Ur2bCxkNtbqjKg7V36Gr8lQbJqmerdNv1w5VV3RcLJW2Bl8/qGqR3QOwyOoE+/wERSp01F3vrfeOWzLpP868NEhyDUsTpxFEWFpEJGmMVvIlawxWjXyaUtoYNKZ5nmSDVsvzY/tz5tMwmKd2TPNWnEQtDvSbtt05aNvtljsLQl+M2RHzZyGFFQ1LoGoBv1ictX4eSrogEaTxWetzFrUa6zUcghMPp2iJ8glrB3gDI9YC32kAo6Hy0TQ+4IUky5wG6GODT2IhlfFpABBy1HySk+YcFtJFo1jA4sboA6jZLCXHLQE32oSHLBkYNXoX0BuSlVA7UYuQjsA0Y9mtGGHzZWP0LPWCMLx932kAMGFO1C2PW3DarVMDI3ZO+L7gRohUNcoN5CwYbGP0t3/+k3Y2u9H+9o//XRGioMBEaBcfQ6xsebXZKMrO3QdBwAXYbsBSgJRPmgF+z7yDl7AKfsCnehSWPSseb93CT0kE2aRHGqMX8Pj1z/h8KzR4ihgg3+GPZqRQW2ck1XKhBxVZgqd3ctYjUbSbsf/zL3//r9rJ1z9DHCE/m7ue02hb7cboe6zP6O5zAEznkEOdQoUV+uRWsCMOdUZ8pgU3f7TvYDsANxVoCGn2rVh7fatz2Bj1Wvu3grStwzbqeRq4t1M3huqwMTohqgX+NOMh7d7N99MFOjYQ5508L3AIAdT2K34KP4MRm9M3ITHwBNwdDdnf/vRvm3BVDGho3P8CsfepBGv5Xr+WXsn+SpVjbue2O/Lfenga8m7MFMR5PWhrba2DKZfEfHhYOo7quDwabY9joN0eFUGr8sM4xoPnLR6pdNA8Em+oPo6nLK8552cQSpswSLZtn9tfDu7hBIwkVQC2kF5D3QPmzsKv/zFh8Q7HzOV3RUO0pT/OaEhzthtmCuSBaf/DX7QTktSiRuWci59a8BVREg4sAq/2SCOZ9uHV+9NTbZyySNN5NNSHxfzqbeoG+QmIMgVGrVWgIr56crJVA4U4q2B5efbh+zNQYMiCd6EI+VTWqsDqy1+8f3by9h7rFbg6gveMRafxPEhZDKaW70RCq/mstbFAYNuDo2a59o2TOSOfeTM+A3JKl+cgKy9nqZGZQwn1/euz03NnBd58cKELa/14TV17Erq6VQygjoJJ1AdFAAhuiolLi4d+QJPw2C8HZZDHYYzychRjOgwREdTl4DRAGggMfCQQycXouqD0d69PTp3VJo0D/eXZcw1CNk2ZBoOaARHNY9pj7bfvTl+au88gFskRzYBllvaS+Sy3tA88S/ucKSvB8dhjdwGrvn/+d9hq1mih9prxQwyppKX9EKeUhJb2JoDEjMHiPU0ut90QPAQsfo6/FIK5GDFv56QgLwuUjSpgyCMwdDObuZ8B8u3z3wJUzkM4U7b9CDqV0gw8aM+3szwE0PMPZ5oc/fpfsdZ7AfVMpJARMr/zsd+WJJy9fVGwtd/+tgbV/djpK1DRbFlAdvrfchoUDRjoIhXUQMW16Qx0AHBVqjDQ35CEQOzU+BjMqToBRIM317B3Q5CEum7gWhjQcECDY0Gc8PjRDORdSn14TfGA88Cn7CNkH0FMUjtK9mHpH3BM6ABfVIJNwJp6kGSrYPD+1//kEMD9jwnmQMRO4gkAvI4gkMV4AD6piUl9XdphysVOUyem18KH2T9S9+XZezlurEiMFxQByQZ5OqMWSoimcwqKfA3R6/lsPKYpn1qD2RbooJrM3wULGr7HvqDxhuRTrPMNcfdRzVhd0xyqi84h1hlBHNP0R3k/Ao+vuJWq6NksT2bgJyGYnmNL2hGkn79/+bwarBCDU6cgyQQIlpDPTk7Pvw/CKPA+VHMKJTzIwrBNY96idvCEw73SY2EBY8MqWHAyhQLKKJf64PRCikClG8ug5KUKh8/x3TCHfNyusgQFhJ/CaC/E/U6JKYlSGilg7968P33zErCkBD1nQQT2shAc3DFURA5fZaO7Fjvj+g3nbJgWVG77pi3MnRYbglqq20GASLDtOKcnJIJNDcgcK2m1FGkhuo7VbbdL2r085KhqIc6ADaydvANwyfwXUE2i5LgIcBhr8veQR+R019h5QkFeHbtbbDxJA5W1L+H1FQ0TUO5O29qHFHnRpb2OeOh297teIRkIagYuLo8wDhlTTeUNzaaGwh4oQuhLyiKap0uj24bjm5YidqlUWDeR0FBvPdYm70Wz1Mb8CFNLe+E0udm8e93qFnNQNwHrBR5x9IpQDlJSOqVRoBD6Cl6zZApMPkPhgGKN+T9w5B78s79v2QfqqXF5pb6zWEH1omg8k3ADVxcwzGLbI1leIxHGhD3ZEUnQwNHQgT37hxb+R90XYMtt0yC6e9vDQ0Jg246KABYBgj2ISpo3A1HNwtCKwN3IR8JRZM5qDUGeeVeqxeE7mAHk2Mroc7boweAr0KdDC9NKoTkCG+bRiIrMSSB0+xwOVh5AZFR8vErKjMI2fcwE+KSSccEsH0cOvaAeBLX0HSiBof/S7ksYuFmLo2xNwnzc0s2hIIvvoGzMYZB141nMmaTxToNxhW1ETZB85qxko2BwUUnduuha+1bv0qokAtpkdey2hfK/tIqegVzU9X2rZ8OyvgUKgMvcsX8Iy+y+ZR/hUrEMU/+ycSDWkvYBGBVAXDT3YWO++KD9pI17AkYwJP6vWF/2ERRie7DHRc/qW80Orh2P+/0nTyyUrQ1Ed/rWmIQZrOVdBWWdvW9dtGFZX6wSpgPrDvi+YtH64upyCERzG0D3jSx+RRfG2UX7UpgG9rBjKB2XztlF55IPFaUktwnbts8uupemHc3CPEjC5blHQpIaryBIBtEm1t6lGFax7nMScLSGuNl99MrCf5u9R4AMzbu2rH8pbUhJ4GH4yeVTEUIGqO/bIK8VFAd852BsPMB15mrHHo/tA+xplhrmgq0ZJrb47Vnig18Eh5cGix9ZCgEVhYj3DQvwGlf0NEryZWk77paF4t6IB4rHOQQpajBntAJimB1k6KXN4uX8CoOUL8bAzq54BhtLAgARAzZHkFzQ5xiSMcVBIocuDCdLQOFWwyZmAOFSUL0Pc1FFv7nmpSsFzYBtJjIo/ATuEuzemyB/ZjEw03DN9RpzKYjewFm2UPg8BkugBrlxwDZo6HS63KYF9yVHPQcxTWh+wrvzSlD7Ay/KgMWmlRVAPEvbBTJ85YhcjyyMzF5akAMI0QiXp04uLATI7BvzUcc+avGZnMSYENhjNn9UBD/Ibk2IgCUS4pCbchK7ZtShoTrAQcmGAXj24jGHgeTbIKaA91hmUPORb3n2spqVIzePS5hNeHEkSDrAOwE3hPTwAhF2jSlJHb8FiQ9/HePbo754ETr2LmWfRSgTMjVErlMqoGohWE5lYCIYz3Knbc2d9vAuNf/uuwfMnkGC/gI8p429EjAvkQQ5lXoN54+diU1y2N0FZcwqVnmgd/kwx2nexHoqf8VEqze4Y1mrhwpYyukeygKw3xg6v/fRTVskjx/enDmfvllxfvOk2MghKWVnDD+QOoet44lhrjV4+PrneDILWab99S/aN6v5DqD5139PIQDQ7NhNR9+sQOcA6HvIBnyjC9Nf/0nDweWuwZvaYPTpMVqzgZTavEzMnn4S+ypDa1FBZp8Gug7mT7Jl7GmlKOkCKrDMGJsrkIAUSuqQaxJAXklzb2qMrRWIZ8r8gf7q9NkLvTJnKHiu1h5BIBCoHORRZ73e29wom7JrAy9NlMjtkdjPHN5OucCpyyFq1DgIKc9khlDDGjLF1dhYgJugVzyzsacE6TZXHH48dFNKrtbS2eOYKSgq5IkJBsiTN05t2Td1dLwG1Sv1mMijy2wEf57hOQxECDbN1ZqCFHISmgpWrE9O5Ad1n7Y7uVpNdTrt9iPKcVO/VSBbf/tJ6ClsALZkrkRYgxKJzSkfWcNMlettACgTw42EcC2CEBgPXzG8OyCxrUyZ7crwGRRuM8i6ohPYAQobLvYhDy6ADoXpOI6OLSv9y5fidRroJo8UvNTYqFjOwc36kFdVlYjoSkMiduR63oEFzJtMY5plA7vft/CbxlC8tfuCc7WtZQ9NxxPde0fgCUiSV7mysaBs+mQNsUzsVGX2KJjhZtiKYu5loiC2l5zdpUtaNp0olnVTNQjuEwu6+gggrUqEYVUiDFHteAe85p10XSoP+EJs0HBwO6TxJAepSkzl8Z8VIG9wojxFYQa85e9c6O8pkITNpxPRB8InlkJNDA/nJJz5RD7l/OE5mCV/eENi3pd6BYUyH3jHMvrxg34pNqmRmLE0NwxiuaYzMvjGwrm/BTu38aNV8/HRkfnt0VFzY9atzUr6K5fhoctQdzJXkokXHl956XCu2F4YJM/4DEZLjqXgg+uUrRUP3EtOZR/A0EVPXscsSbV9gRkMBryLRw39I5xd41Ashn2gtDPgmOh6DAFq1sSptm5cmatx4ILyQiKXT4ungXpESOklVnE0Gb2x6cGxoXsFt3kODkJa7G3OQE1IixWynYYEo4sBeotPJZ0HEkad5HcsZxBeCh+lc4ah+dD8R0gCDATGbKAGjJcOJIgzAV6GY3FrtOXAN829cja8s64/1fnnKPpAxysyfQjHqzpDxwf7bbOGvCIDjVsHpde3KaiMbnP3p/qxW97MjzC+nxCfaNyRaY42i0UstnHmBQM9BlZoEb2BbWNLC4nGu7U46DOwHFsfcBXYPBJuIhrEEF/At9M0YfI2cqQ/5lBnnFjDHAi/IHJtqLJ59QxCPxO9S5nJDW8PkIJtgGL3tcizMISl/GMIrbAJSJTTUwJZgeuMXIWlOZtMQqEGlmvj7Swogj2Hk/G8oJ5j1k4hE5OMUtHzeUMSdLa/KOWMnGdpSpYAx3+NKjyYT6tnsKRB9SZda/TddxG3L8COtPAURIyYJh/ApF4MWPpD/XFRHE94cVykgKaMIjJjuoD6GhdfYiPKMC5iy7sEJ/HpmF8ZB8VVr3Jt+83KW+P1ZwDpY7wWt4GfTPszgzqBJ3my0YReXoQPha/oTuKCGVC2SM8SX6JtPCBF1iTSEMTw3Xfy4YHjEFM+Qwnh07ez3LB75pDY2H7PDZOPvo5x0OYbcSFxMshP6ZD6RdR9dUh1vY4Tb/ldYLVStUivw2Jedj/AoCcPK5hxnTkXl7doVV2VrqHKmGVTg5kgSXgpiL12Rte2+Oa9cHrX5q1dgAeFsn75srlBlbpyxdtqERj30eHBRfV8WXE0ckYR34U7BQfYIRP6dbldqR/Xm23lqlyz6jPPSRZ427nboXvojsdWuZ3IqpS/DRADStPZ8mmST39Mg5wORDFhllH5us4mkYpe26JT/5YnLV1ITdF3X6OVrcueK/d3ji4bhbpyZ7KtisWH+5taWIRwLAMo+EmWQNGckAmPvVUH1XGlCshkQY4zx98d5SDkiyBzN0kKNQtntNgVXjkmk9sxM/3NYCam0fWUG8H46RweEAivawydn1C3MEv5PyLHvJPXF+KLqcvbeI2ErEr5VVEj3Axl6/vs4/3EPvVbHJEhYWhMWX5nQhNgNlqQ5mGoxm+OoPb42d0sCZo725OevbBE43EBdcSrR3b7CTaEisR9s7eUA3huL+Hfm8cA3O+Lho7aHMo3ejyVGxBUXBALsmTlZDYkBQFkvxZwA4PV72aRixWnbOVZ9KfEcGemoMqCtwuUJAHVqJaEqcC3pHBFrJBZXCXIu4T+oP6+U/xqEKqDm2uxj5enG/sUGdiekvTW6RBeInYe3CMtHu6EUelCt65EPEmXTM3v2Hgred+x8RbM5sZ7ZRhRexHVlaNsb8v7Tl59VoBlzLJ5JPgAPlua4i4Y/PZd/AVjV72xU9ohwjzubHr8ynC6imwloomoFNkypjn8IpUHI9l4v2/jZvj/TXRBYWdYP4AMu0hzoTfTbcMp78fFg8GPZYEfKpwYubV6JjomjVPICZwdt+ygThjif3h/ZugBfirSws9GcIXPrmMsUxzdEx/H8Q9KYIKTZaCWb4c0/HjnhoqYtkI3SfjXAs7uLwTubonf96OQza9Oyn7LGWOQ41dm5+cOv/LFsPCChjkxZO0OpYZawvvbvvo27uPeewZvv/KNQO2wt8uvhUV301D7u3oaTD7yfvHnjFfc/Bd4WXZ2q4AA1Wl5ofzWRebYcxJCSSt60CT1psGcZV++rGTSttHB3fpUzfp1389Z6ldT6idS9U+iLk3YVXyOsIsqeZIruswM/GLO3AAX3AJtiIIM3AvEMSzZ4FdUbLLRLcxTwJZddVO0qgUuyZNfGh6rZovomCvR8dLOICk3gIZaa1y594gcNGP5PbJ+Py5sddp/hlmPVWvFl6ppgsgvxpePP2nH/O8foYgdF4mFDYdOwEJM/OsMPvkJfJPaFiPcj27QzslEeh9I5L+OVrWsHO/cX2D/PTcG8FPnFFX/HYEYnxlgzkRmYTjKeY1VyNz5PVq5oc9188sXPceLByRZ3n/cJsO5o0qGA4NiBLFvXDkjsfbqtrWwDW/By4As9xd5A2apHd3cmUiIekEmt/Kjrbwgn+YkQe7hTUheMNrl+lV9ob7VGaU5DGz0lkRrbl2amLgjwj6QiZlS+Qc6xy3+lznHLfE/P/hfWqCMWQ=="}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
# Si ya habia una version cargada, se cambia por la nueva SIN perder los
# modelos que ya estan en la GPU (antes Colab seguia usando el codigo viejo).
_old = sys.modules.get("ap3d.engine")
_eng = getattr(_old, "_ENGINE", None)
_app = sys.modules.get("ap3d.app")
if _app is not None:
    for _k in ("inline", "web"):
        try:
            _app._SERVERS.pop(_k).close()
        except Exception:
            pass
    try:
        _app._SERVERS.pop("tunnel").terminate()
    except Exception:
        pass
for _m in [m for m in sys.modules if m == "ap3d" or m.startswith("ap3d.")]:
    del sys.modules[_m]
if _eng is not None:
    from ap3d import engine as _e
    _eng.__class__ = _e.Engine
    _eng._ensure()
    _e._ENGINE = _eng
    print("Código actualizado; los modelos siguen cargados en la GPU.")
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias, TripoSG y Blender 4.5 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2, TripoSG y Blender 4.5 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
INSTALAR_TRIPOSG = True  # @param {type:"boolean"}
# @markdown TripoSG es el modelo abierto del equipo de **Tripo AI**: formas con más detalle y bordes limpios. Se suma como segundo motor; si no instala, sigue Hunyuan3D.
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' 'diffusers==0.32.2' 'transformers==4.49.0' 'accelerate>=1.2,<1.6' "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender: 4.5 LTS (el mas nuevo estable); si no se puede, 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    for ver, serie in (("4.5.3", "4.5"), ("4.5.2", "4.5"), ("4.5.1", "4.5"), ("4.2.3", "4.2")):
        url = f"https://download.blender.org/release/Blender{serie}/blender-{ver}-linux-x64.tar.xz"
        r = subprocess.run(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz "
                           f"-C /content && mv /content/blender-{ver}-linux-x64 /content/blender",
                           shell=True)
        subprocess.run("rm -f /content/blender.tar.xz", shell=True)
        if r.returncode == 0 and os.path.exists("/content/blender/blender"):
            break

# --- TripoSG (modelo abierto del equipo de Tripo AI): forma de alta fidelidad
if INSTALAR_TRIPOSG and not os.path.exists("/content/TripoSG"):
    sh("git clone -q --depth 1 https://github.com/VAST-AI-Research/TripoSG /content/TripoSG", check=False)
    sh(f"{sys.executable} -m pip install -q peft jaxtyping typeguard scikit-image", check=False)
    r = subprocess.run(f"{sys.executable} -m pip install -q diso", shell=True,
                       capture_output=True, text=True)
    print("TripoSG:", "listo" if r.returncode == 0 else "diso no compilo (TripoSG desactivado)")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
MODO_CALIDAD_TAMBIEN = True  # @param {type:"boolean"}
CARGAR_TRIPOSG = True  # @param {type:"boolean"}
CARGAR_TEXTO_A_3D = True  # @param {type:"boolean"}
# @markdown Turbo (5 pasos + FlashVDM) siempre se carga: la forma sale en segundos. Con la casilla de arriba también queda cargado el modelo de 50 pasos para el preajuste "Maxima".
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
# Hunyuan3D-2 necesita transformers 4.49: con versiones nuevas su codificador
# de imagen (DINOv2) no carga ("Missing key(s) ... main_image_encoder").
import importlib.metadata as _md, subprocess
if _md.version("transformers") != "4.49.0" or not _md.version("diffusers").startswith("0.32"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.49.0",
                    "diffusers==0.32.2", "accelerate>=1.2,<1.6"], check=True)
    if "transformers" in sys.modules:
        raise SystemExit("Versiones corregidas. Ve a Entorno de ejecución › Reiniciar sesión "
                         "y vuelve a ejecutar la celda 1 y después esta celda 3.")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0-turbo/*"] + (["hunyuan3d-dit-v2-0/*"] if MODO_CALIDAD_TAMBIEN else [])
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0-turbo/*", "hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv-turbo/*"]
                      + (["hunyuan3d-dit-v2-mv/*"] if MODO_CALIDAD_TAMBIEN else []), max_workers=16)
if CARGAR_TRIPOSG and os.path.exists("/content/TripoSG"):
    try:
        snapshot_download("VAST-AI/TripoSG", local_dir="/content/TripoSG/pretrained_weights/TripoSG", max_workers=16)
        snapshot_download("briaai/RMBG-1.4", local_dir="/content/TripoSG/pretrained_weights/RMBG-1.4", max_workers=16)
    except Exception as e:
        print("TripoSG no se pudo descargar:", e)
        CARGAR_TRIPOSG = False
if CARGAR_TEXTO_A_3D:
    try:
        snapshot_download("Tencent-Hunyuan/HunyuanDiT-v1.1-Diffusers-Distilled", max_workers=16)
    except Exception as e:
        print("Texto a 3D no disponible:", e)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA, want_turbo=True,
         want_quality=MODO_CALIDAD_TAMBIEN)
if CARGAR_TRIPOSG:
    eng.load_triposg()
print("Todo listo. Sigue con la celda 4 o 5.")

## 3b. Prueba rápida sin página web (opcional)

In [ ]:
# @title 3b. Prueba rápida SIN página web (confirma que el motor funciona y mide tiempos)
import glob, sys, time
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
_img = sorted(glob.glob("/content/Hunyuan3D-2/assets/example_images/*.png"))[0]
_t = time.time()
_r = get_engine().run(_img, steps=5, octree=384, faces=100000, tex=2048, rig="auto",
                      progress=lambda f, m: print(f"{int(f*100):3d}%  {m}", flush=True) if int(f*100) % 10 == 0 else None)
_i = _r["info"]
print(f"\nLISTO en {time.time()-_t:.0f}s  ->  {_r['zip']}")
print("forma", _i.get("segundos_forma"), "s · textura", _i.get("segundos_textura"),
      "s · Blender", _i.get("segundos_blender"), "s · avisos:", _i.get("avisos"))

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()